# 🔀 Routing: Directing Inputs to Specialized Handlers

## Learning Objectives
In this notebook, you will learn:
1. **The routing pattern** - Classify an input once, then send it down exactly one specialist path
2. **A constrained classifier** - Force the routing decision into a `Literal` schema with `with_structured_output`
3. **Specialist handlers** - One node per category, each with its own focused prompt
4. **Conditional edges as the router** - Map the stored decision to the next node, with a safe default
5. **Testing the router** - Check that each request reaches the handler you expect

## Where this fits
`modules/05_prompt_chaining` ran every step in a fixed order. Routing adds one decision up front: *which* chain should handle this input. `02_routing_by_model_tier.ipynb` routes on a different axis (which model), and `03_routing_lcel_runnable_branch.ipynb` rebuilds this router in LCEL.

## Prerequisites
- LangGraph `StateGraph` and conditional edges — `modules/05_prompt_chaining/notebooks/01_prompt_chaining_langgraph.ipynb`
- `with_structured_output` — `modules/04_langchain_agents_and_middleware/notebooks/03_structured_output.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (no other keys needed)

---
## 📖 Part 1: What is Routing?

Routing is a fundamental agentic workflow pattern that involves intelligently classifying incoming inputs and directing them to the most appropriate specialized process or handler. Think of it as a sophisticated traffic controller that ensures each request reaches the right destination based on its characteristics and requirements.

This pattern operates on the principle that different types of queries require different handling strategies. Rather than using a one-size-fits-all approach, routing ensures that each input is processed by the most suitable component, leading to improved efficiency, accuracy, and resource utilization.

| Property | Value |
|---|---|
| Origin | Anthropic, *Building Effective Agents* (Dec 2024) — [anthropic.com/research/building-effective-agents](https://www.anthropic.com/research/building-effective-agents), "Routing" section |

```
                                   ┌──▶ generate_story ──┐
user request ──▶ classify request ─┼──▶ generate_poem  ──┼──▶ output
                                   └──▶ generate_joke  ──┘
                                        (exactly ONE runs)
```

### Key Concepts:
- **`add_conditional_edges`**: maps a node's output (or a condition function's result) to one of several next nodes
- **State**: stores the routing decision, so the condition function only has to read it
- **Condition function**: pure Python that turns the decision into a node name

> **Key Insight**: The LLM makes one small decision — a label from a fixed set. Everything after that is ordinary, testable Python dispatch.

---
## 📦 Step 1: Environment Setup

Load the API key, then create the chat model through the course factory.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

✅ Environment ready


In [2]:
# ============================================================================
# LLM INITIALIZATION: One model for the classifier and the handlers
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

✅ LLM initialized: gpt-4o-mini


---
## 🏷️ Part 2: The Routing Schema

A router is only as reliable as its classifier. A free-text answer like *"I think they want a poem"* is useless as a routing key, so the decision is constrained to a `Literal` of exactly three values: the model *cannot* return a label your graph has no edge for.

In [3]:
# ============================================================================
# ROUTING SCHEMA: Constrain the classifier to three valid destinations
# ============================================================================
from typing import Literal

from pydantic import BaseModel, Field


class ContentRoute(BaseModel):
    """Schema for routing creative content requests"""
    content_type: Literal["story", "poem", "joke"] = Field(
        description="The type of creative content to generate"
    )

# Create a structured router using the schema
content_router = llm.with_structured_output(ContentRoute)

print(f"✅ Classifier ready — labels: {ContentRoute.model_fields['content_type'].annotation}")

✅ Classifier ready — labels: typing.Literal['story', 'poem', 'joke']


### 2.1 🗂️ The Workflow State

The state carries the request in, the decision across, and the output out.

In [4]:
# ============================================================================
# STATE: Request in, decision across, output out
# ============================================================================
from typing_extensions import TypedDict


class RoutingState(TypedDict):
    user_input: str          # Original user request
    routing_decision: str    # Decision made by router
    final_output: str        # Generated content

print(f"✅ RoutingState fields: {list(RoutingState.__annotations__)}")

✅ RoutingState fields: ['user_input', 'routing_decision', 'final_output']


---
## ✍️ Part 3: Specialist Handlers

Each category gets its own node with its own prompt. This is the payoff of routing: every prompt can be tuned for one kind of request without compromising the others.

In [5]:
# ============================================================================
# SPECIALIST HANDLERS: One node per content type
# ============================================================================
def generate_story(state: RoutingState) -> dict:
    """Specialized handler for story generation"""
    story_prompt = f"Write an engaging short story based on: {state['user_input']}"
    response = llm.invoke(story_prompt)
    return {"final_output": response.content}

def generate_poem(state: RoutingState) -> dict:
    """Specialized handler for poem generation"""
    poem_prompt = f"Create a creative poem inspired by: {state['user_input']}"
    response = llm.invoke(poem_prompt)
    return {"final_output": response.content}

def generate_joke(state: RoutingState) -> dict:
    """Specialized handler for joke generation"""
    joke_prompt = f"Write a funny, clean joke about: {state['user_input']}"
    response = llm.invoke(joke_prompt)
    return {"final_output": response.content}


print("✅ 3 handlers defined")

✅ 3 handlers defined


---
## 🧭 Part 4: The Router

The router is two pieces, like the gate in module 05: a **node** that calls the classifier and stores the decision, and a **condition function** that turns the stored decision into a node name. The condition function uses `.get(..., "generate_story")` as a safe default, so even an unexpected label still lands somewhere.

In [6]:
# ============================================================================
# ROUTER NODE: Ask the classifier, store the decision
# ============================================================================
from langchain_core.messages import HumanMessage, SystemMessage


def route_content_request(state: RoutingState) -> dict:
    """Analyze input and determine the appropriate content type"""
    routing_messages = [
        SystemMessage(
            content="Analyze the user's request and determine if they want a story, poem, or joke. "
                   "Consider keywords, tone, and intent in your decision."
        ),
        HumanMessage(content=state["user_input"])
    ]

    decision = content_router.invoke(routing_messages)
    return {"routing_decision": decision.content_type}

# ============================================================================
# CONDITION FUNCTION: Decision → next node name (pure Python, no LLM call)
# ============================================================================
def determine_next_step(state: RoutingState) -> str:
    """Conditional function to route to the appropriate handler"""
    routing_map = {
        "story": "generate_story",
        "poem": "generate_poem",
        "joke": "generate_joke",
    }
    return routing_map.get(state["routing_decision"], "generate_story")


print("✅ Router node and condition function defined")

✅ Router node and condition function defined


### 4.1 🏗️ Building the Workflow

One fixed edge into the router, one conditional edge out of it with three targets, and every handler goes straight to `END`.

In [7]:
# ============================================================================
# BUILD THE GRAPH: classify → (one of three handlers) → END
# ============================================================================
from langgraph.graph import END, START, StateGraph

# Create the routing workflow graph
routing_graph = StateGraph(RoutingState)

# Add all processing nodes
routing_graph.add_node("route_content_request", route_content_request)
routing_graph.add_node("generate_story", generate_story)
routing_graph.add_node("generate_poem", generate_poem)
routing_graph.add_node("generate_joke", generate_joke)

# Define the workflow connections
routing_graph.add_edge(START, "route_content_request")

# Add conditional routing based on decision
routing_graph.add_conditional_edges(
    "route_content_request",
    determine_next_step,
    {
        "generate_story": "generate_story",
        "generate_poem": "generate_poem",
        "generate_joke": "generate_joke"
    },
)

# Connect all handlers to the end
routing_graph.add_edge("generate_story", END)
routing_graph.add_edge("generate_poem", END)
routing_graph.add_edge("generate_joke", END)

# Compile the workflow
content_routing_workflow = routing_graph.compile()

print("✅ Routing workflow compiled")

✅ Routing workflow compiled


In [8]:
# ============================================================================
# INSPECT THE GRAPH: Print nodes and edges (no network call needed)
# ============================================================================
g = content_routing_workflow.get_graph()

print("🧩 Nodes:", [n for n in g.nodes if not n.startswith("__")])
print("🔗 Edges:")
for e in g.edges:
    kind = "conditional" if e.conditional else "fixed"
    print(f"   {e.source:>18} ──▶ {e.target:<18} ({kind})")

🧩 Nodes: ['route_content_request', 'generate_story', 'generate_poem', 'generate_joke']
🔗 Edges:
            __start__ ──▶ route_content_request (fixed)
   route_content_request ──▶ generate_joke      (conditional)
   route_content_request ──▶ generate_poem      (conditional)
   route_content_request ──▶ generate_story     (conditional)
        generate_joke ──▶ __end__            (fixed)
        generate_poem ──▶ __end__            (fixed)
       generate_story ──▶ __end__            (fixed)


---
## 🧪 Part 5: Testing the Routing System

Three requests, each worded so that the category is implied rather than named ("something funny", "a tale", "something poetic"). The classifier has to read intent, not match keywords.

In [9]:
# ============================================================================
# TEST: One request per expected destination
# ============================================================================
test_cases = [
    ("Tell me something funny about artificial intelligence", "joke"),
    ("I want to hear a tale about a brave knight", "story"),
    ("Create something poetic about the ocean", "poem"),
]

for user_request, expected in test_cases:
    print(f"\n--- Processing: '{user_request}' ---")
    result = content_routing_workflow.invoke({"user_input": user_request})
    mark = "✅" if result["routing_decision"] == expected else "❌"
    print(f"{mark} Routed to: {result['routing_decision']} (expected {expected})")
    print(f"📄 Output: {result['final_output'][:150]}...")


--- Processing: 'Tell me something funny about artificial intelligence' ---
✅ Routed to: joke (expected joke)
📄 Output: Why did the artificial intelligence break up with its partner?

Because it just couldn't find the right "algorithm" for love!...

--- Processing: 'I want to hear a tale about a brave knight' ---
✅ Routed to: story (expected story)
📄 Output: Once upon a time, in the verdant kingdom of Eldoria, there lived a brave knight named Sir Cedric. He was renowned not only for his skill with a sword ...

--- Processing: 'Create something poetic about the ocean' ---
✅ Routed to: poem (expected poem)
📄 Output: **Whispers of the Ocean**

In the cradle of the azure deep,  
Where secrets of the ages sleep,  
The ocean sings a timeless song,  
A melody both fier...


Each request cost exactly two LLM calls — one to classify, one in the chosen handler — no matter how many handlers exist. Adding a fourth category means one more `Literal` value, one more node and one more edge; the other handlers are untouched.

---
## 📝 Summary

In this notebook, we learned:

### 1. The routing pattern
- Classify once, then run exactly one specialist path; separation of concerns lets each prompt specialise
- Cost per request is constant: one classification plus one handler call

### 2. A constrained classifier
- `with_structured_output` + a `Literal` field guarantees the decision is a label the graph can route

### 3. Specialist handlers
- One node per category, each with its own prompt, all writing the same state key

### 4. Conditional edges as the router
- Router node stores the decision; a pure condition function maps it to a node name, with a default

### 5. Testing the router
- Test with phrasing that implies the category; check the decision, not only the output

### Next Steps
- **`02_routing_by_model_tier.ipynb`**: route on a different axis — send each request to a cheap or a reasoning *model*, and measure whether it pays